# Cross-Sectional Analyses: Cancer vs Healthy Donors

- Cross-sectional comparisons were performed between **cancer participants** and **healthy donor controls**.
- Analyses evaluated differences in proteomic features at defined study time points.
- Group-level statistical testing using MWU was conducted to identify significantly altered features between cancer and healthy cohorts.
- Results are summarized and BH multiple testing correction are applied where applicable, and visualized using comparative plots.


In [7]:
require(data.table)
require(ggplot2)
require(dplyr)
require(table1)
require(tidyr)
require(parallel)
require(ggridges)
require(pheatmap)
require(UpSetR)
require(ComplexHeatmap)
source('helper_functions_ndmm.R')

In [8]:
plasma <- readRDS('../../data/olink/MM_Plasma_Olink_Final.rds')

In [9]:
### First split the healthy subjects  
healthy = plasma[plasma$cohort.cohortGuid %in% c('BR1','BR2')]

### then extract only Flu Year 1 day 0 samples 
healthy = healthy[sample.visitDetails =='Flu_Y1D0']
unique_samples_boolean = length(unique(healthy$subject.subjectGuid)) == length(unique(healthy$sample.sampleKitGuid))
print(paste('The # of unique subjects and unique samples is equal:', unique_samples_boolean))

mm_treatment = plasma[sample.visitDetails %in% c('PreTx','PI2C','EI','ASCT60d', 'ASCT1y','ASCT2y')]

healthy$Cohort = 'Healthy'
mm_treatment$Cohort = 'NDMM'  
plasma_treatment = rbind(healthy, mm_treatment)
plasma_treatment[plasma_treatment$sample.visitDetails =='Flu_Y1D0']$sample.visitDetails <- 'Healthy'

[1] "The # of unique subjects and unique samples is equal: TRUE"


# Helper Functions for Cross-sectional Comparisons Against Healthy Donors

## Note on Analyses 
In these longitudinal analyses, the following short-hand naming convention is used to denote the different treatment timepoints in the multiple myeloma study:

- Pre-Tx = Pre-Treatment (Plasma & BMIF)
- PI2C = Post Induction 2 Cycles (Plasma only)
- EI = End Induction (Plasma & BMIF)
- ASCT60d = Post Transplant 60 Days (Plasma only)
- ASCT90d = Post Transplant 90 Days (BMIF only)
- ASCT1y = Post Transplant 1 year (Plasma & BMIF)
- ASCT2y = Post Transplant 2 year (Plasma & BMIF)

For the healthy comparisons, we compare each above time point against healthy, matched donors.

## Helper Functions

In [10]:
### write function to test proteins 
test_protein <- function(assay, visit, plasma_treatment, plot=F, size=4){
  
  ### subset matrix to protein of interest
  tmp = plasma_treatment[olink.assay ==assay & sample.visitDetails %in% c(visit, 'Healthy')]
  
  ### filter to the columns of interest
  dat = tmp[,c('olink.NPX_norm','Cohort','sample.visitDetails', 'subject.subjectGuid')]
  
  ### it looks like there are 
  ### duplicate runs 
  ### per subject + visit combo
  ### so the choice is to take mean
  dat <- dat[, list(NPX = mean(olink.NPX_norm),
                    Visit = first(sample.visitDetails),
                    Cohort = first(Cohort)),
             by=subject.subjectGuid]
  
  dat = drop_na(dat)
  if(length(table(dat$Cohort))<2){
       
        res = data.frame(
            Assay = assay,
            Pvalue = NA,
            Log2FC = NA,
            N = NA)

      } else{
        ### test for differences 
        ### Using a MWU paired test 
        res = data.frame(
            Assay = assay,
            Pvalue = suppressWarnings(wilcox.test(dat$NPX ~ dat$Cohort)$p.value),
            Log2FC = mean(dat[Cohort=='NDMM']$NPX) - mean(dat[Cohort=='Healthy']$NPX),
            N = nrow(dat)
        )
      
  }  ### add (optional) plotting function 
      if(plot){
        
        p=ggplot(dat,
               aes(x=Visit,
                   y=NPX,
                  col=Cohort,
                   fill=Cohort
                  ))+geom_boxplot()+geom_point()+
          ggpubr::stat_compare_means(method='wilcox.test',size=size)+
          ggtitle(paste(res[1,]))
        #print(p)
        return(p)
        } else{ 
        return(res)
        }     
}

In [11]:
#### summarize contrast 
summarize_contrast <- function(proteins, t1, plasma_treatment){
 
 ### Run differential test across all proteins 
 results <- lapply(
      proteins,
      function(x) {
        tryCatch(
           test_protein(x, t1,plasma_treatment), silent=T)
          }
    )
  ### Remove proteins that
  ### failed differential testing
  results = rbindlist(results[sapply(results, class) !='try-error'])
  
  ### transform p-values into q-values 
  results$adjP = p.adjust(results$Pvalue, method='fdr')  
  results$Time1 = t1
  results$Time2 = 'Healthy'
  results$Contrast = paste(t1, 'Healthy', sep='-')
  return(results)
}


# Plasma Cross-sectional Differentials across different visits

In [12]:
### Define Contrasts of Interest 
proteins <- unique(plasma_treatment$olink.assay)
comparisons <- c('PreTx','PI2C','EI','ASCT60d', 'ASCT1y','ASCT2y')

## Run healthy comparisons

In [13]:
plasma_healthy_comparisons <- mclapply(comparisons,
       function(x){
           print(x)
           summarize_contrast(proteins, x, plasma_treatment)}, mc.cores=length(comparisons) )

In [14]:
plasma_healthy_comparisons <- rbindlist(plasma_healthy_comparisons)

In [15]:
plasma_healthy_comparisons[, sum(adjP < 0.05, na.rm=T), by=Contrast]
length(unique(plasma_healthy_comparisons[adjP < 0.05]$Assay))

Contrast,V1
<chr>,<int>
PreTx-Healthy,689
PI2C-Healthy,587
EI-Healthy,409
ASCT60d-Healthy,540
ASCT1y-Healthy,523
ASCT2y-Healthy,332


[1] 1050

In [16]:
plasma_healthy_comparisons$Contrast = factor(plasma_healthy_comparisons$Contrast,
                                                  levels =c(
                                                      'PreTx-Healthy',
                                                      'PI2C-Healthy',
                                                      'EI-Healthy',
                                                      'ASCT60d-Healthy',
                                                      'ASCT1y-Healthy',
                                                      'ASCT2y-Healthy')
                                                  )

## Save Plasma Results 

In [17]:
write.csv(plasma_healthy_comparisons,
          file='../../data/olink/output/plasma_longitudinal_comparisons_to_healthy.csv')